# Notebook: Portfolio Optimization & Black–Litterman

ใช้ Python 3 standard library และกด Run All ตามลำดับได้ ตัวเลขทุกชุดเป็นข้อมูลสมมติสำหรับเรียนรู้ ไม่ใช่ข้อมูลตลาดหรือคำแนะนำการลงทุน ภาพ SVG ฝังอยู่ใน Notebook แล้ว

# Portfolio Optimization & Black–Litterman

เราจะบอกคอมพิวเตอร์อย่างไรว่า “พอร์ตที่ดีที่สุด” หมายถึงอะไร?

บท [Portfolio Theory](../portfolio-theory.html) วางเกณฑ์ไว้แล้วว่าเราสนใจผลตอบแทนคาดหวัง ความเสี่ยง และการเคลื่อนไหวร่วมกันของสินทรัพย์ บทนี้นำเกณฑ์เหล่านั้นมาเขียนเป็นโจทย์ที่คำนวณน้ำหนักพอร์ตได้จริง

Optimizer ไม่ได้รู้เองว่าควรลดความเสี่ยง เพิ่มผลตอบแทน ห้ามขายชอร์ต หรือเกาะ benchmark แค่ไหน เราต้องกำหนด objective function, decision variables, ข้อมูลที่ป้อน และ constraints ให้ครบ คำตอบที่ได้จึงผูกกับโจทย์นั้นทุกบรรทัด

เราจะใช้ตัวอย่างสินทรัพย์สมมติสี่ตัวต่อเนื่องทั้งบท แล้วดูว่า Black–Litterman ผสมพอร์ตตลาดกับมุมมองของผู้ลงทุนอย่างไร ตัวเลขไม่มีชื่อสินทรัพย์ ช่วงวันที่ หรือข้อมูลตลาดจริง จึงใช้เพื่อเรียนรู้กลไกเท่านั้น

In [1]:
import itertools
import math

def close(actual, expected, tol=1e-9):
    assert math.isclose(actual, expected, rel_tol=tol, abs_tol=tol), (actual, expected)

def dot(a, b):
    return sum(x*y for x, y in zip(a, b))

def transpose(a):
    return [list(column) for column in zip(*a)]

def matmul(a, b):
    bt = transpose(b)
    return [[dot(row, column) for column in bt] for row in a]

def matvec(a, x):
    return [dot(row, x) for row in a]

def solve(a, b):
    augmented = [list(row) + [value] for row, value in zip(a, b)]
    n = len(augmented)
    assert all(len(row) == n + 1 for row in augmented)
    for column in range(n):
        pivot = max(range(column, n), key=lambda row: abs(augmented[row][column]))
        assert abs(augmented[pivot][column]) > 1e-14, "singular system"
        augmented[column], augmented[pivot] = augmented[pivot], augmented[column]
        scale = augmented[column][column]
        augmented[column] = [value/scale for value in augmented[column]]
        for row in range(n):
            if row == column:
                continue
            factor = augmented[row][column]
            augmented[row] = [left-factor*right for left, right in zip(augmented[row], augmented[column])]
    return [row[-1] for row in augmented]

def variance(weights, covariance):
    return dot(weights, matvec(covariance, weights))

MU = [.05, .07, .15, .27]
SD = [.07, .12, .30, .60]
CORR = [
    [1, .8, .5, .4],
    [.8, 1, .7, .5],
    [.5, .7, 1, .8],
    [.4, .5, .8, 1],
]
SIGMA = [[CORR[i][j]*SD[i]*SD[j] for j in range(4)] for i in range(4)]
ONE = [1.0]*4
RF = .025
print("Hypothetical one-year simple returns. Python standard library only.")

Hypothetical one-year simple returns. Python standard library only.


## เขียนโจทย์ให้ครบก่อนกด Solve

โจทย์ optimization ทั่วไปเขียนได้ว่า

$$
\begin{aligned}
\min_{\mathbf x}\quad &f(\mathbf x)\\
\text{subject to}\quad
&g_j(\mathbf x)\le b_j,\\
&h_k(\mathbf x)=c_k.
\end{aligned}
$$

องค์ประกอบมีสามส่วน

- \(f\) คือ [objective function](../glossary.html#objective-function) ที่ต้องการให้ต่ำสุดหรือสูงสุด เช่น variance ของพอร์ต
- \(\mathbf x\) คือเวกเตอร์ [decision variables](../glossary.html#decision-variable) เช่น น้ำหนักสินทรัพย์แต่ละตัว
- \(g_j(\mathbf x)\) และ \(h_k(\mathbf x)\) คือ [constraints](../glossary.html#optimization-constraint) เช่น น้ำหนักรวม 100%, ห้ามน้ำหนักติดลบ หรือ tracking error ไม่เกินเพดาน

การเปลี่ยนโจทย์ \(\max f(\mathbf x)\) เป็น \(\min[-f(\mathbf x)]\) ไม่เปลี่ยนจุดที่เหมาะสม และการคูณ objective ด้วยค่าบวกพร้อมบวกค่าคงที่ก็ไม่เปลี่ยน \(\mathbf x^*\) แต่ค่าของ objective หลังแปลงต้องแปลงกลับก่อนตีความ



Objective เดิมไม่ได้รับประกันคำตอบเดิม เมื่อ feasible set เปลี่ยน จุดที่ดีที่สุดอาจย้ายจากก้นแอ่งไปอยู่บนเส้นหรือขอบเขตที่อนุญาต

ข้อจำกัดต้องใช้หน่วยเดียวกับข้อมูล ตัวอย่างเช่น \(\mu\) ต่อปีต้องจับคู่กับ covariance ต่อปี และ target return 10% ต้องเขียนเป็น 0.10 หากสูตรใช้หน่วยทศนิยม การป้อน 10 ลงในสูตรเดียวกันจะเปลี่ยนโจทย์ไปหนึ่งร้อยเท่า

## เห็น minimum ผ่าน gradient และ Hessian

เริ่มจากโจทย์ที่ยังไม่มี constraints ถ้า \(f\) เรียบและจุดเหมาะสมอยู่ภายในโดเมน [gradient](../glossary.html#gradient) ที่จุดนั้นต้องเป็นศูนย์

$$
\nabla f(\mathbf x^*)=\mathbf 0.
$$

เงื่อนไขนี้คัดจุดที่เป็นไปได้ แต่ยังแยก minimum, maximum และ saddle point ไม่ได้ [Hessian](../glossary.html#hessian) รวบรวมอนุพันธ์อันดับสองของ \(f\)

$$
H_f(\mathbf x)=
\left[
\frac{\partial^2 f}{\partial x_i\partial x_j}
\right]_{i,j}.
$$

ถ้า Hessian เป็น positive definite ที่ stationary point ฟังก์ชันโค้งขึ้นทุกทิศทางและจุดนั้นเป็น strict local minimum ถ้าเป็น negative definite จะได้ strict local maximum ส่วน Hessian ที่มีทั้งทิศบวกและลบชี้ไปที่ saddle point



ลูกศร gradient หายไปที่จุดกึ่งกลางทั้งสองภาพ ฝั่งซ้ายโค้งขึ้นทุกทิศทาง ส่วนฝั่งขวามีทิศหนึ่งโค้งขึ้นและอีกทิศโค้งลง

### Mean–variance แบบไม่มี constraint บนน้ำหนักสินทรัพย์เสี่ยง

ให้ \(r\) เป็นผลตอบแทนสินทรัพย์ปลอดความเสี่ยง \(\boldsymbol\mu\) เป็นเวกเตอร์ผลตอบแทนคาดหวังของสินทรัพย์เสี่ยง และ \(\Sigma\) เป็น covariance matrix น้ำหนักที่ไม่ได้ลงในสินทรัพย์เสี่ยงจะอยู่ในสินทรัพย์ปลอดความเสี่ยง โจทย์จึงเขียนเป็น

$$
\max_{\mathbf w}
\left[
r+\mathbf w^\top(\boldsymbol\mu-r\mathbf 1)
-\frac{\lambda}{2}\mathbf w^\top\Sigma\mathbf w
\right],
\qquad \lambda>0.
$$

First-order condition ให้

$$
\boldsymbol\mu-r\mathbf 1-\lambda\Sigma\mathbf w^*=\mathbf 0.
$$

เมื่อ \(\Sigma\) กลับด้านได้

$$
\boxed{
\mathbf w^*=\frac{1}{\lambda}
\Sigma^{-1}(\boldsymbol\mu-r\mathbf 1).
}
$$

Hessian ของ objective คือ \(-\lambda\Sigma\) ถ้า \(\Sigma\) positive definite ฟังก์ชันจะ concave อย่างเคร่งครัดและคำตอบนี้เป็น global maximum เพียงจุดเดียว ค่า \(\lambda\) เพิ่มสองเท่าจะลดน้ำหนักสินทรัพย์เสี่ยงทุกตัวลงครึ่งหนึ่งในโจทย์นี้ โดยน้ำหนักส่วนที่เหลือย้ายไปสินทรัพย์ปลอดความเสี่ยง

สูตรนี้ยังไม่ได้บังคับ long-only เพดานน้ำหนัก เงินกู้ หลักประกัน หรือต้นทุนซื้อขาย หากผลลัพธ์มีน้ำหนักรวม 160% จะหมายถึงกู้ 60% ที่อัตรา \(r\) ภายใต้สมมติฐานของแบบจำลอง

## สร้าง covariance matrix จาก volatility และ correlation

ตัวอย่างหลักใช้สินทรัพย์สมมติ \(X_1,\ldots,X_4\) และผลตอบแทน simple return ระยะหนึ่งปี

| สินทรัพย์ | \(X_1\) | \(X_2\) | \(X_3\) | \(X_4\) |
|---|---:|---:|---:|---:|
| ผลตอบแทนคาดหวัง \(\mu_i\) | 5% | 7% | 15% | 27% |
| ส่วนเบี่ยงเบนมาตรฐาน \(\sigma_i\) | 7% | 12% | 30% | 60% |

Correlation matrix คือ

$$
R=
\begin{pmatrix}
1&0.8&0.5&0.4\\
0.8&1&0.7&0.5\\
0.5&0.7&1&0.8\\
0.4&0.5&0.8&1
\end{pmatrix}.
$$

สร้างเมทริกซ์แนวทแยง \(S=\operatorname{diag}(0.07,0.12,0.30,0.60)\) แล้วคำนวณ

$$
\boxed{\Sigma=SRS.}
$$

เพราะ \(S\) เป็นเมทริกซ์แนวทแยง เราจึงมี \(S^\top=S\) ค่าแนวทแยงของ \(\Sigma\) คือ variance และค่านอกแนวทแยงคือ covariance

$$
\Sigma=
\begin{pmatrix}
0.0049&0.00672&0.0105&0.0168\\
0.00672&0.0144&0.0252&0.0360\\
0.0105&0.0252&0.0900&0.1440\\
0.0168&0.0360&0.1440&0.3600
\end{pmatrix}.
$$



สีเข้มช่วยเปรียบเทียบขนาดภายในแต่ละเมทริกซ์ ส่วนตัวเลขเป็นค่าที่ใช้คำนวณจริง Correlation ไม่มีหน่วย ขณะที่ covariance ใช้หน่วยผลตอบแทนยกกำลังสอง

Covariance matrix ต้องเป็น positive semidefinite เพราะ \(\mathbf w^\top\Sigma\mathbf w\) คือ variance และห้ามติดลบ หากเมทริกซ์เกือบ singular การกลับเมทริกซ์โดยตรงจะขยายความคลาดเคลื่อน ควรแก้ระบบสมการและตรวจ condition number แทน

In [2]:
expected_sigma = [
    [.0049, .00672, .0105, .0168],
    [.00672, .0144, .0252, .036],
    [.0105, .0252, .09, .144],
    [.0168, .036, .144, .36],
]
for row, expected in zip(SIGMA, expected_sigma):
    for actual, target in zip(row, expected):
        close(actual, target, 1e-12)
print("Sigma = diag(SD) @ Corr @ diag(SD)")
for row in SIGMA:
    print("  ", [round(value, 6) for value in row])

Sigma = diag(SD) @ Corr @ diag(SD)
   [0.0049, 0.00672, 0.0105, 0.0168]
   [0.00672, 0.0144, 0.0252, 0.036]
   [0.0105, 0.0252, 0.09, 0.144]
   [0.0168, 0.036, 0.144, 0.36]


## Regression ก็เริ่มจาก objective

การประมาณ beta หรือ factor exposure เป็นโจทย์ optimization อีกแบบ ให้ \(Y\) เป็นผลตอบแทนสินทรัพย์ \(X\) เป็นเมทริกซ์ปัจจัย และ \(\boldsymbol\beta\) เป็นสัมประสิทธิ์

$$
Y=X\boldsymbol\beta+\boldsymbol\varepsilon.
$$

[Ordinary Least Squares](../glossary.html#ordinary-least-squares) เลือก \(\boldsymbol\beta\) เพื่อลดผลรวม residual ยกกำลังสอง

$$
\min_{\boldsymbol\beta}
(Y-X\boldsymbol\beta)^\top(Y-X\boldsymbol\beta).
$$

First-order condition ให้ normal equations

$$
X^\top X\widehat{\boldsymbol\beta}=X^\top Y.
$$

ถ้า \(X\) มีคอลัมน์เป็นอิสระเชิงเส้นครบ

$$
\widehat{\boldsymbol\beta}_{\mathrm{OLS}}
=(X^\top X)^{-1}X^\top Y.
$$



Residual บวกและลบหักล้างกันได้เมื่อรวมตรง ๆ OLS จึงรวมกำลังสองของระยะแต่ละจุด เส้นนี้เป็นตัวอย่างเชิงสอน ไม่ใช่ regression จากหลักทรัพย์จริง

Regression ที่ได้ beta จากข้อมูลย้อนหลังยังไม่พิสูจน์ CAPM ค่า beta และ alpha เปลี่ยนตามช่วงข้อมูล ความถี่ ตัวแทนตลาด และตัวแปรที่ใส่ใน \(X\)

### GLS เมื่อ residual มี covariance ไม่เท่ากับ \(s^2I\)

ให้

$$
\mathbb E[\boldsymbol\varepsilon\mid X]=\mathbf 0,
\qquad
\operatorname{Var}(\boldsymbol\varepsilon\mid X)=\Omega,
$$

โดย \(\Omega\) positive definite [Generalized Least Squares](../glossary.html#generalized-least-squares) ใช้ระยะ Mahalanobis

$$
\min_{\boldsymbol\beta}
(Y-X\boldsymbol\beta)^\top
\Omega^{-1}(Y-X\boldsymbol\beta),
$$

จึงได้

$$
\widehat{\boldsymbol\beta}_{\mathrm{GLS}}
=(X^\top\Omega^{-1}X)^{-1}X^\top\Omega^{-1}Y.
$$

ถ้า \(\Omega=CC^\top\) จาก Cholesky decomposition ให้ \(Y^*=C^{-1}Y\), \(X^*=C^{-1}X\) และ \(\boldsymbol\varepsilon^*=C^{-1}\boldsymbol\varepsilon\) จะได้

$$
\operatorname{Var}(\boldsymbol\varepsilon^*\mid X)
=C^{-1}\Omega C^{-\top}=I.
$$

จากนั้นใช้ OLS กับ \(Y^*=X^*\boldsymbol\beta+\boldsymbol\varepsilon^*\) ได้คำตอบ GLS การแปลงนี้หมุนและปรับสเกล residual ให้ covariance เป็นเอกลักษณ์



วงรีแสดง residual ที่มี scale และ correlation ต่างกัน หลัง whitening ระยะ Euclidean ในพิกัดใหม่เท่ากับ Mahalanobis distance ในพิกัดเดิม

ในงานจริง \(\Omega\) มักต้องประมาณ จึงได้ Feasible GLS ความแม่นของผลลัพธ์ขึ้นกับแบบจำลอง covariance ที่เลือกด้วย

In [3]:
# A small deterministic regression example with an intercept.
x = [-2, -1, 0, 1, 2]
y = [-2.7, -1.0, 1.2, 2.8, 5.1]
X = [[value, 1.0] for value in x]
Xt = transpose(X)
beta_ols = solve(matmul(Xt, X), matvec(Xt, y))
residuals = [actual-predicted for actual, predicted in zip(y, matvec(X, beta_ols))]
print(f"OLS slope={beta_ols[0]:.6f}, intercept={beta_ols[1]:.6f}, SSE={dot(residuals,residuals):.6f}")

# GLS with a known residual covariance. Compute Omega^-1 X and Omega^-1 y by solving systems.
OMEGA = [
    [1.0, .25, 0, 0, 0],
    [.25, 1.5, .2, 0, 0],
    [0, .2, .8, .15, 0],
    [0, 0, .15, 1.2, .25],
    [0, 0, 0, .25, 1.8],
]
omega_inv_y = solve(OMEGA, y)
omega_inv_X_columns = [solve(OMEGA, column) for column in transpose(X)]
omega_inv_X = transpose(omega_inv_X_columns)
beta_gls = solve(matmul(Xt, omega_inv_X), matvec(Xt, omega_inv_y))
print(f"GLS slope={beta_gls[0]:.6f}, intercept={beta_gls[1]:.6f}")
assert all(math.isfinite(value) for value in beta_ols + beta_gls)

OLS slope=1.940000, intercept=1.080000, SSE=0.112000
GLS slope=1.921736, intercept=1.104756


## Lagrange ใส่ equality constraints เข้าไปในสมการ

สมมติโจทย์มีข้อจำกัด \(g_j(\mathbf x)=b_j\) เราสร้าง [Lagrangian](../glossary.html#lagrange-multiplier)

$$
\mathcal L(\mathbf x,\boldsymbol\lambda)
=f(\mathbf x)
+\sum_{j=1}^{m}\lambda_j[g_j(\mathbf x)-b_j].
$$

แล้วแก้ระบบ

$$
\nabla_{\mathbf x}\mathcal L=\mathbf 0,
\qquad
\frac{\partial\mathcal L}{\partial\lambda_j}
=g_j(\mathbf x)-b_j=0.
$$

ที่จุดสัมผัส gradient ของ objective ขนานกับ gradient ของ constraint ให้ \(v^*(\mathbf b)\) เป็นค่า objective ที่เหมาะสม ภายใต้เงื่อนไข regularity และ Lagrangian แบบที่เขียนข้างบน เรามี \(\partial v^*/\partial b_j=-\lambda_j\) เครื่องหมายจะเปลี่ยนหากตั้ง Lagrangian คนละแบบ



จุดที่ต่ำกว่านี้อยู่นอกเส้น constraint ส่วนจุดอื่นบนเส้นเดียวกันตัด contour ระดับสูงกว่า จุดสัมผัสจึงแก้ทั้ง stationarity และ constraint พร้อมกัน

## ลด variance โดยกำหนดผลตอบแทนเป้าหมาย

ให้ \(m\) เป็นผลตอบแทนคาดหวังเป้าหมาย และอนุญาตให้ขายชอร์ตได้ โจทย์ risky-only คือ

$$
\min_{\mathbf w}\frac12\mathbf w^\top\Sigma\mathbf w
$$

subject to

$$
\boldsymbol\mu^\top\mathbf w=m,
\qquad
\mathbf 1^\top\mathbf w=1.
$$

ตั้ง

$$
\mathcal L=\frac12\mathbf w^\top\Sigma\mathbf w
-\lambda(\boldsymbol\mu^\top\mathbf w-m)
-\gamma(\mathbf 1^\top\mathbf w-1).
$$

First-order condition ให้

$$
\Sigma\mathbf w-\lambda\boldsymbol\mu-\gamma\mathbf 1=\mathbf 0,
$$

ดังนั้น

$$
\mathbf w^*=\Sigma^{-1}
(\lambda\boldsymbol\mu+\gamma\mathbf 1).
$$

กำหนดสเกลาร์

$$
A=\mathbf 1^\top\Sigma^{-1}\mathbf 1,
\quad
B=\boldsymbol\mu^\top\Sigma^{-1}\mathbf 1,
\quad
C=\boldsymbol\mu^\top\Sigma^{-1}\boldsymbol\mu.
$$

เมื่อ \(AC-B^2>0\)

$$
\lambda=\frac{Am-B}{AC-B^2},
\qquad
\gamma=\frac{C-Bm}{AC-B^2}.
$$

ตัวอย่างสี่สินทรัพย์ให้

$$
A\approx239.3440,
\quad B\approx9.61846,
\quad C\approx0.550281.
$$

เมื่อ \(m=10\%\) จะได้ \(\lambda\approx0.365279\), \(\gamma\approx-0.0105013\) และ

$$
\boxed{
\mathbf w^*\approx
\begin{pmatrix}
0.528412\\
0.172888\\
0.159764\\
0.138935
\end{pmatrix}.}
$$

น้ำหนักรวม 100% ผลตอบแทนคาดหวัง 10% และ volatility ประมาณ 16.1328%



คำตอบนี้เป็นบวกทุกตัวโดยบังเอิญ ขอบเขตของโจทย์ยังอนุญาตให้ short และคำตอบเปลี่ยนทันทีเมื่อ target หรือ inputs เปลี่ยน

ในโค้ดควรแก้ระบบสมการเชิงเส้นหรือ KKT system แทนการสร้าง \(\Sigma^{-1}\) เต็มก้อน แล้วตรวจผลย้อนกลับว่า \(\mathbf 1^\top\mathbf w=1\) และ \(\boldsymbol\mu^\top\mathbf w=m\) ภายใน tolerance ที่กำหนด

In [4]:
inverse_one = solve(SIGMA, ONE)
inverse_mu = solve(SIGMA, MU)
A = dot(ONE, inverse_one)
B = dot(MU, inverse_one)
C = dot(MU, inverse_mu)
D = A*C-B*B

def minimum_variance_target(target, indices=range(4)):
    indices = list(indices)
    cov = [[SIGMA[i][j] for j in indices] for i in indices]
    means = [MU[i] for i in indices]
    ones = [1.0]*len(indices)
    inv_one = solve(cov, ones)
    inv_mu = solve(cov, means)
    a, b, c = dot(ones, inv_one), dot(means, inv_one), dot(means, inv_mu)
    d = a*c-b*b
    budget_multiplier = (c-b*target)/d
    return_multiplier = (a*target-b)/d
    weights = [budget_multiplier*x + return_multiplier*y for x, y in zip(inv_one, inv_mu)]
    return weights, budget_multiplier, return_multiplier

w10, gamma, lam = minimum_variance_target(.10)
expected = [.528412108337758, .172888075206605, .159764342703102, .138935473752535]
for actual, target in zip(w10, expected):
    close(actual, target)
close(sum(w10), 1)
close(dot(w10, MU), .10)
close(gamma, -.010501299717696)
close(lam, .365279369427286)
print("Target 10% weights:", [f"{100*w:.4f}%" for w in w10])
print(f"Expected return={100*dot(w10,MU):.4f}%, volatility={100*math.sqrt(variance(w10,SIGMA)):.4f}%")

Target 10% weights: ['52.8412%', '17.2888%', '15.9764%', '13.8935%']
Expected return=10.0000%, volatility=16.1328%


## จาก target หนึ่งค่าไปสู่ทั้ง frontier

เมื่อปล่อยให้ \(m\) เปลี่ยนไป variance ต่ำสุดของแต่ละ target คือ

$$
\boxed{
\sigma_P^2(m)=
\frac{Am^2-2Bm+C}{AC-B^2}.
}
$$

Global minimum-variance portfolio อยู่ที่

$$
m_{\mathrm{GMV}}=\frac{B}{A},
\qquad
\mathbf w_{\mathrm{GMV}}
=\frac{\Sigma^{-1}\mathbf 1}{A},
\qquad
\sigma_{\mathrm{GMV}}^2=\frac1A.
$$

สำหรับตัวอย่างนี้

$$
\mathbf w_{\mathrm{GMV}}\approx
\begin{pmatrix}
1.274887\\
-0.263113\\
0.016339\\
-0.028113
\end{pmatrix},
$$

ผลตอบแทนคาดหวังประมาณ 4.0187% และ volatility 6.4638% น้ำหนัก \(X_1\) 127.49% มาจากการ short \(X_2\) และ \(X_4\) โจทย์คณิตศาสตร์ยอมรับคำตอบนี้เพราะเรายังไม่ได้ห้าม short หรือกำหนด gross exposure



ครึ่งบนเหนือ GMV เป็น efficient branch ภายใต้ชุดสินทรัพย์และการอนุญาต short นี้ จุด target 10% เป็นคำตอบของ equality-constrained problem หนึ่งค่า



เมื่อ Target สูงขึ้น Optimizer ปรับ long–short หลายขาเพื่อให้ budget และ target เท่ากันพร้อมลด variance จึงต้องอ่านการเปลี่ยนแปลงของน้ำหนักทุกขาร่วมกัน

### เพิ่มสินทรัพย์ปลอดความเสี่ยง

เมื่อ \(r=2.5\%\) และต้องการผลตอบแทน \(m\) น้ำหนักสินทรัพย์เสี่ยงที่ลด variance คือ

$$
\mathbf w^*(m)=
\frac{(m-r)\Sigma^{-1}(\boldsymbol\mu-r\mathbf 1)}
{(\boldsymbol\mu-r\mathbf 1)^\top
\Sigma^{-1}(\boldsymbol\mu-r\mathbf 1)}.
$$

ที่ \(m=10\%\)

$$
\mathbf w^*\approx
\begin{pmatrix}
0.887352\\
0.081263\\
0.154843\\
0.121649
\end{pmatrix}.
$$

น้ำหนักสินทรัพย์เสี่ยงรวม 124.5108% น้ำหนักสินทรัพย์ปลอดความเสี่ยงจึงเป็น \(1-1.245108=-24.5108\%\) ซึ่งแปลว่ากู้เงินภายใต้สมมติฐานที่กู้ได้ที่ 2.5% โดยไม่มีข้อจำกัด

Tangency portfolio ของสินทรัพย์เสี่ยง normalize จากทิศทางเดียวกัน

$$
\mathbf w_T=
\frac{\Sigma^{-1}(\boldsymbol\mu-r\mathbf 1)}
{\mathbf 1^\top\Sigma^{-1}(\boldsymbol\mu-r\mathbf 1)}
\approx
\begin{pmatrix}
0.712671\\
0.065266\\
0.124361\\
0.097701
\end{pmatrix}.
$$

พอร์ตนี้มีผลตอบแทนคาดหวังประมาณ 8.5236%, volatility 12.8731% และ Sharpe ratio ประมาณ 0.4679 ดูความหมายของ frontier, CAL และ tangency เพิ่มเติมได้ใน [บท Portfolio Theory](../portfolio-theory.html#tangency)

In [5]:
w_gmv = [value/A for value in inverse_one]
close(sum(w_gmv), 1)
close(dot(w_gmv, MU), B/A)
close(variance(w_gmv, SIGMA), 1/A)
print("GMV weights:", [f"{100*w:.4f}%" for w in w_gmv])
print(f"GMV return={100*dot(w_gmv,MU):.4f}%, volatility={100*math.sqrt(variance(w_gmv,SIGMA)):.4f}%")

excess = [value-RF for value in MU]
direction = solve(SIGMA, excess)
w_tangency = [value/sum(direction) for value in direction]
tangency_return = dot(w_tangency, MU)
tangency_sd = math.sqrt(variance(w_tangency, SIGMA))
print("Tangency weights:", [f"{100*w:.4f}%" for w in w_tangency])
print(f"Return={100*tangency_return:.4f}%, volatility={100*tangency_sd:.4f}%, Sharpe={(tangency_return-RF)/tangency_sd:.6f}")

target = .10
risky_scale = (target-RF)/(tangency_return-RF)
w_risky = [risky_scale*value for value in w_tangency]
w_safe = 1-sum(w_risky)
close(w_safe, -.245107788138614)
print(f"Funded target 10%: risky total={100*sum(w_risky):.4f}%, risk-free={100*w_safe:.4f}%")

GMV weights: ['127.4887%', '-26.3113%', '1.6339%', '-2.8113%']
GMV return=4.0187%, volatility=6.4638%
Tangency weights: ['71.2671%', '6.5266%', '12.4361%', '9.7701%']
Return=8.5236%, volatility=12.8731%, Sharpe=0.467919
Funded target 10%: risky total=124.5108%, risk-free=-24.5108%


## Black–Litterman ตั้งต้นจากพอร์ตตลาด

น้ำหนัก mean–variance ไวต่อ \(\boldsymbol\mu\) มาก ผลตอบแทนคาดหวังที่ต่างกันไม่กี่จุดอาจเปลี่ยน long–short positions ขนาดใหญ่ [Black–Litterman](../glossary.html#black-litterman) เริ่มจากน้ำหนักตลาดที่สังเกตได้ แล้วใช้ reverse optimization หา expected excess returns ที่สอดคล้องกับน้ำหนักนั้น ก่อนผสม views ที่ระบุพร้อมความไม่แน่นอน



Roadmap แยกข้อมูลตลาด ความเห็น และการจัดสรรออกจากกัน จึงย้อนตรวจได้ว่าการเปลี่ยนน้ำหนักมาจาก input ส่วนใด [ดาวน์โหลดไฟล์ Excalidraw ที่แก้ไขต่อได้](assets/diagrams/optimization-black-litterman-roadmap.excalidraw)

### 1. Reverse optimization หา prior

สมมติพอร์ตตลาดมีน้ำหนัก

$$
\mathbf w_{\mathrm{mkt}}=
\begin{pmatrix}
0.05&0.40&0.45&0.10
\end{pmatrix}^{\!\top}.
$$

จากคำตอบ mean–variance \(\mathbf w^*=\lambda^{-1}\Sigma^{-1}\widetilde{\boldsymbol\mu}\) เราแก้ย้อนกลับเพื่อหา implied equilibrium excess returns

$$
\boxed{
\boldsymbol\Pi=
\lambda_{\mathrm{mkt}}\Sigma\mathbf w_{\mathrm{mkt}}.
}
$$

ตัวอย่างกำหนด market Sharpe ratio เท่ากับ 0.5 พอร์ตตลาดมี volatility ประมาณ 22.3523% จึงใช้

$$
\lambda_{\mathrm{mkt}}
=\frac{\operatorname{SR}_{\mathrm{mkt}}}
{\sigma_{\mathrm{mkt}}}
\approx2.24.
$$

ผลที่คำนวณด้วย \(\lambda_{\mathrm{mkt}}=2.24\) คือ

$$
\boldsymbol\Pi\approx
\begin{pmatrix}
0.020917\\
0.047121\\
0.146731\\
0.259930
\end{pmatrix}.
$$

เราเขียน prior ของ excess returns เป็น

$$
\widetilde{\boldsymbol\mu}
\sim N(\boldsymbol\Pi,\tau\Sigma).
$$

ตัวอย่างกำหนด \(\tau=1/120\) เป็น teaching convention เท่านั้น เลข 120 จำลองกรณีมีผลตอบแทนรายเดือนสิบปี แต่ไม่ได้อ้างว่า \(\Sigma\) ชุดนี้ประมาณจากข้อมูลดังกล่าว วิธีตั้ง \(\tau\) ไม่มีคำตอบเดียวและต้องสอดคล้องกับนิยามของ \(\Omega\) ที่ใช้กับ views

### 2. เขียน views เป็น \(P,Q,\Omega\)

กำหนดสอง views

1. \(X_3\) จะให้ excess return สูงกว่า \(X_1\) อยู่ 10 จุดเปอร์เซ็นต์
2. \(X_2\) จะให้ excess return 3%

จึงได้

$$
P=
\begin{pmatrix}
-1&0&1&0\\
0&1&0&0
\end{pmatrix},
\qquad
Q=
\begin{pmatrix}
0.10\\0.03
\end{pmatrix}.
$$

แถวแรกของ \(P\) รวมเป็นศูนย์เพราะเป็น relative view ส่วนแถวที่สองเลือก \(X_2\) ตัวเดียวและเป็น absolute view เขียนแบบจำลองของ views เป็น

$$
Q=P\widetilde{\boldsymbol\mu}+\boldsymbol\varepsilon_v,
\qquad
\boldsymbol\varepsilon_v\sim N(\mathbf 0,\Omega).
$$

ตัวอย่างใช้

$$
\Omega_{ii}=\left[P(\tau\Sigma)P^\top\right]_{ii},
\qquad
\Omega_{ij}=0\quad(i\ne j).
$$

ค่าแนวทแยงเล็กลงทำให้ view มีน้ำหนักมากขึ้น ค่าใหญ่ขึ้นทำให้ posterior อยู่ใกล้ prior มากขึ้น การเรียกค่าหนึ่งว่า “มั่นใจ 80%” ต้องมี mapping เพิ่มเติม เช่นวิธีของ Idzorek จึงไม่ควรติดป้ายเปอร์เซ็นต์ให้ \(\Omega\) โดยตรง

### 3. ผสมเป็น posterior

Posterior mean เขียนในรูปที่คำนวณได้เสถียรว่า

$$
\boxed{
\widehat{\boldsymbol\mu}_{\mathrm{BL}}
=\boldsymbol\Pi
+\tau\Sigma P^\top
\left(P\tau\Sigma P^\top+\Omega\right)^{-1}
(Q-P\boldsymbol\Pi).
}
$$

สูตรนี้เทียบเท่ากับ precision form

$$
\widehat{\boldsymbol\mu}_{\mathrm{BL}}
=\left[(\tau\Sigma)^{-1}+P^\top\Omega^{-1}P\right]^{-1}
\left[(\tau\Sigma)^{-1}\boldsymbol\Pi+P^\top\Omega^{-1}Q\right].
$$

สำหรับตัวอย่าง

$$
\widehat{\boldsymbol\mu}_{\mathrm{BL}}
\approx
\begin{pmatrix}
0.016782\\
0.037552\\
0.124843\\
0.227174
\end{pmatrix}.
$$



Posterior ขยับทั้งสี่สินทรัพย์เพราะ covariance เชื่อม views เข้ากับสินทรัพย์อื่น เส้นทางการขยับจึงไม่ได้จำกัดอยู่ที่ช่องของ \(P\) เท่านั้น

### 4. แปลง posterior เป็นน้ำหนัก

เมื่อใช้ risk aversion \(\lambda\)

$$
\mathbf w_{\mathrm{BL}}
=\frac1\lambda
\Sigma^{-1}\widehat{\boldsymbol\mu}_{\mathrm{BL}}.
$$

ที่ \(\lambda=2.24\) น้ำหนักสินทรัพย์เสี่ยงและสินทรัพย์ปลอดความเสี่ยงเป็น

| น้ำหนัก | \(X_1\) | \(X_2\) | \(X_3\) | \(X_4\) | สินทรัพย์ปลอดความเสี่ยง |
|---|---:|---:|---:|---:|---:|
| พอร์ตตลาด | 5.00% | 40.00% | 45.00% | 10.00% | 0.00% |
| Black–Litterman | 9.87% | 16.59% | 40.13% | 10.00% | 23.41% |



Risk aversion เปลี่ยนขนาดรวมของ risky allocation ส่วน \(P,Q,\Omega\) และ covariance เปลี่ยนทิศทางสัมพัทธ์ของน้ำหนัก ใน convention นี้ \(\Omega\) scale พร้อม \(\tau\) ทำให้ \(\tau\) หักล้างจาก posterior mean; ถ้ากำหนด \(\Omega\) แยกต่างหาก \(\tau\) จะมีผล

Posterior covariance ของค่าเฉลี่ยและ covariance ของผลตอบแทนเป็นคนละวัตถุ สูตรจัดพอร์ตข้างบนใช้ \(\Sigma\) เป็น covariance ของผลตอบแทน การนำ posterior uncertainty ไปบวกหรือใช้แทน \(\Sigma\) เป็นอีก convention ที่ต้องประกาศให้ชัด

In [6]:
MARKET = [.05, .40, .45, .10]
LAMBDA_MARKET = 2.24
TAU = 1/120
P = [[-1, 0, 1, 0], [0, 1, 0, 0]]
Q = [.10, .03]
prior = [LAMBDA_MARKET*value for value in matvec(SIGMA, MARKET)]
tau_sigma = [[TAU*value for value in row] for row in SIGMA]
projected = matmul(matmul(P, tau_sigma), transpose(P))
omega = [[projected[i][i] if i == j else 0.0 for j in range(2)] for i in range(2)]
system = [[projected[i][j]+omega[i][j] for j in range(2)] for i in range(2)]
innovation = [q-p for q, p in zip(Q, matvec(P, prior))]
view_update = solve(system, innovation)
posterior_adjustment = matvec(matmul(tau_sigma, transpose(P)), view_update)
posterior = [base+change for base, change in zip(prior, posterior_adjustment)]
expected_posterior = [.016781811019405, .037552435573995, .124842704859425, .227173738447279]
for actual, target in zip(posterior, expected_posterior):
    close(actual, target)
weights = [value/LAMBDA_MARKET for value in solve(SIGMA, posterior)]
risk_free = 1-sum(weights)
close(risk_free, .234140487785057)
print("Prior excess returns:", [f"{100*x:.4f}%" for x in prior])
print("Posterior excess returns:", [f"{100*x:.4f}%" for x in posterior])
print("Posterior risky weights:", [f"{100*x:.4f}%" for x in weights])
print(f"Risk-free weight={100*risk_free:.4f}%")

Prior excess returns: ['2.0917%', '4.7121%', '14.6731%', '25.9930%']
Posterior excess returns: ['1.6782%', '3.7552%', '12.4843%', '22.7174%']
Posterior risky weights: ['9.8696%', '16.5860%', '40.1304%', '10.0000%']
Risk-free weight=23.4140%


## KKT บอกว่า constraint ใดกำลังบังคับคำตอบ

ข้อจำกัด long-only เขียนเป็น \(-w_i\le0\) เพดานน้ำหนักเขียนเป็น \(w_i-u_i\le0\) และข้อจำกัด gross exposure มักต้องเพิ่มตัวแปรช่วยเพื่อจัดการค่าสัมบูรณ์

สำหรับโจทย์

$$
\min_{\mathbf x}f(\mathbf x)
\quad\text{subject to}\quad
g_j(\mathbf x)\le0,
\quad h_k(\mathbf x)=0,
$$

[Karush–Kuhn–Tucker conditions](../glossary.html#kkt-conditions) ประกอบด้วย

$$
\nabla f(\mathbf x^*)
+\sum_j\nu_j\nabla g_j(\mathbf x^*)
+\sum_k\lambda_k\nabla h_k(\mathbf x^*)=\mathbf0,
$$

$$
g_j(\mathbf x^*)\le0,
\qquad h_k(\mathbf x^*)=0,
\qquad \nu_j\ge0,
$$

$$
\boxed{\nu_jg_j(\mathbf x^*)=0.}
$$

Complementary slackness บอกว่า constraint ที่ยังเหลือช่องว่างต้องมี multiplier เป็นศูนย์ ส่วน multiplier บวกเกิดได้เมื่อ constraint ชนขอบ ใน convex quadratic portfolio problem ที่ constraints เป็น affine และมีจุด feasible ตามเงื่อนไข regularity KKT ใช้ยืนยัน global optimum ได้



ในตัวอย่าง target 20% คำตอบ unconstrained มี \(w_1=-71.96\%\) ส่วน long-only ทำให้ constraint \(w_1\ge0\) binding และเลือก \(w_1=0\)

พอร์ต long-only สำหรับ target 20% คือ

$$
\mathbf w_{\mathrm{long\text{-}only}}
\approx
\begin{pmatrix}
0\\0.026316\\0.539474\\0.434211
\end{pmatrix},
$$

มี volatility ประมาณ 40.3829% สูงกว่าคำตอบที่อนุญาต short เล็กน้อย ข้อจำกัดทำให้พอร์ตลงทุนได้ตามกติกา แลกกับ objective ที่แย่ลงหรือเท่าเดิม

### 130–30 บอก net และ gross exposure

พอร์ต 130–30 ถือสถานะ long รวม 130% และ short รวม 30%

$$
\sum_i\max(w_i,0)=1.30,
\qquad
-\sum_i\min(w_i,0)=0.30.
$$

Net exposure เท่ากับ 100% และ gross exposure เท่ากับ 160% ชื่อนี้ยังไม่กำหนดเองว่าควร long หรือ short ตัวใด ต้องมี objective, ข้อจำกัดรายสินทรัพย์ ต้นทุนยืม และกติกา rebalance เพิ่มเติม

In [7]:
# Enumerate active sets for this four-asset teaching example.
def long_only_target(target):
    best = None
    for size in range(2, 5):
        for indices in itertools.combinations(range(4), size):
            try:
                active_weights, _, _ = minimum_variance_target(target, indices)
            except (AssertionError, ZeroDivisionError):
                continue
            if min(active_weights) < -1e-10:
                continue
            full = [0.0]*4
            for index, value in zip(indices, active_weights):
                full[index] = value
            candidate = variance(full, SIGMA)
            if best is None or candidate < best[0]:
                best = candidate, full
    assert best is not None
    return best

long_only_variance, w_long = long_only_target(.20)
expected = [0, .026315789473684, .539473684210526, .434210526315789]
for actual, target in zip(w_long, expected):
    close(actual, target)
close(sum(w_long), 1)
close(dot(w_long, MU), .20)
print("Long-only target 20% weights:", [f"{100*w:.4f}%" for w in w_long])
print(f"Volatility={100*math.sqrt(long_only_variance):.4f}%; w1 constraint is binding.")

Long-only target 20% weights: ['0.0000%', '2.6316%', '53.9474%', '43.4211%']
Volatility=40.3829%; w1 constraint is binding.


## แยก benchmark ออกจาก active positions

ให้ \(\mathbf w_B\) เป็นน้ำหนัก benchmark และ \(\mathbf w_P\) เป็นน้ำหนักพอร์ต กำหนด [active weights](../glossary.html#active-weight)

$$
\Delta\mathbf w=\mathbf w_P-\mathbf w_B.
$$

ถ้าพอร์ตและ benchmark ลงทุนครบ 100% ทั้งคู่

$$
\boxed{\mathbf 1^\top\Delta\mathbf w=0.}
$$

ผลตอบแทนพอร์ตแยกเป็น

$$
R_P=\mathbf w_B^\top\mathbf R
+\Delta\mathbf w^\top\mathbf R
=R_B+R_A.
$$

Tracking error ภายใต้ covariance \(\Sigma\) คือ

$$
\operatorname{TE}
=\sqrt{\Delta\mathbf w^\top\Sigma\Delta\mathbf w}.
$$

ตัวอย่าง active mean–variance problem ใช้

$$
\max_{\Delta\mathbf w}
\left[
\Delta\mathbf w^\top\boldsymbol\alpha
-\frac{\lambda_A}{2}
\Delta\mathbf w^\top\Sigma\Delta\mathbf w
\right]
$$

subject to \(\mathbf 1^\top\Delta\mathbf w=0\) และข้อจำกัดน้ำหนักหรือ tracking error ที่กองทุนใช้จริง



Active weight บวกคือ overweight เทียบ benchmark และค่าลบคือ underweight ค่าลบจะเป็น short ก็ต่อเมื่อน้ำหนักพอร์ตสุดท้าย \(w_{P,i}\) ต่ำกว่าศูนย์ ผลรวม active weights เป็นศูนย์เพราะ benchmark รับ budget 100% ไว้แล้ว

In [8]:
benchmark = [.20, .30, .35, .15]
portfolio = [.15, .36, .39, .10]
active = [p-b for p, b in zip(portfolio, benchmark)]
close(sum(active), 0)
tracking_error = math.sqrt(variance(active, SIGMA))
print("Active weights:", [f"{100*w:+.2f}%" for w in active])
print(f"Net active weight={100*sum(active):.2f}%, tracking error={100*tracking_error:.4f}%")

Active weights: ['-5.00%', '+6.00%', '+4.00%', '-5.00%']
Net active weight=0.00%, tracking error=2.0946%


## ทดลองผสม views กับพอร์ตตลาด

ห้องทดลองใช้ \(\Sigma\), market weights, \(\tau=1/120\), \(P\) และ \(Q\) ชุดเดียวกับตัวอย่าง ปรับ risk aversion และตัวคูณความไม่แน่นอนของ views เพื่อดู posterior expected excess returns กับน้ำหนักพอร์ตพร้อมกัน

ตัวคูณ \(\Omega\) ต่ำทำให้ views มีน้ำหนักมากขึ้น ตัวคูณสูงดึง posterior กลับเข้าหา market-implied prior ส่วน \(\lambda\) เปลี่ยนขนาด risky allocation หลังคำนวณ posterior แล้ว ห้องทดลองไม่ใช้ข้อมูลตลาด ไม่รวมค่าธรรมเนียม ภาษี turnover หรือข้อจำกัด long-only

## ตรวจ optimizer ก่อนเชื่อน้ำหนัก

| จุดที่ต้องตรวจ | วิธีตรวจ |
|---|---|
| หน่วยของ \(\mu,\Sigma,r,Q\) | ใช้ช่วงเวลาและรูปแบบทศนิยมเดียวกันทุกตัว |
| Constraint residuals | คำนวณ \(\mathbf1^\top\mathbf w-1\), \(\boldsymbol\mu^\top\mathbf w-m\) และ inequality violations หลัง solve |
| Covariance เกือบ singular | ตรวจ eigenvalues/condition number ใช้ linear solve, regularization หรือ factor model ที่มีเหตุผลรองรับ |
| น้ำหนักไวต่อ expected returns | ขยับ \(\mu\), views และ \(\Omega\) ทีละน้อย แล้วเทียบ turnover, gross exposure และ objective |
| Solver status | แยก optimal, infeasible, unbounded และ numerical failure ไม่ใช้คำตอบล่าสุดเงียบ ๆ |
| In-sample fit | ทดสอบนอกช่วงที่ใช้ประมาณค่าและคิดต้นทุนซื้อขาย |
| Posterior uncertainty | แยก uncertainty ของค่าประมาณ expected return ออกจาก covariance ของผลตอบแทน |
| Benchmark | ตรวจว่า active weights รวมเป็นศูนย์และใช้ benchmark ตัวเดียวกับรายงานผล |

พอร์ตที่แก้สมการได้ทุกบรรทัดยังแพ้จริงได้ เพราะ \(\mu\), \(\Sigma\), views และข้อจำกัดเป็นแบบจำลองของอนาคต การเพิ่ม precision ทางตัวเลขช่วยให้แก้โจทย์ที่ตั้งไว้ได้แม่นขึ้น แต่ไม่ได้ทำให้ inputs ถูกต้องขึ้นเอง

## ลองคำนวณต่อ

1. ตรวจว่าน้ำหนัก target 10% รวมเป็นหนึ่ง และใช้ \(\boldsymbol\mu^\top\mathbf w\) คำนวณผลตอบแทนกลับ
2. ใช้ \(\gamma=-0.0105013\) ใน \(\Sigma^{-1}(\lambda\boldsymbol\mu+\gamma\mathbf1)\) แล้วสังเกตว่าการเปลี่ยนเครื่องหมาย \(\gamma\) ทำให้ constraints ผิดอย่างไร
3. น้ำหนักสินทรัพย์เสี่ยงสำหรับ target 10% เมื่อมี \(r=2.5\%\) รวมเป็นเท่าไร และต้องถือสินทรัพย์ปลอดความเสี่ยงเท่าไร
4. ทำไมแถว relative view ของ \(P\) จึงรวมเป็นศูนย์ ส่วน absolute view ของสินทรัพย์หนึ่งตัวรวมเป็นหนึ่ง
5. ที่ target 20% long-only solution มี constraint ใด binding และ complementary slackness บอกอะไรเกี่ยวกับ multiplier ของ constraint นั้น
6. ถ้า benchmark และพอร์ตลงทุนครบ 100% ทั้งคู่ จงพิสูจน์ว่า active weights รวมเป็นศูนย์

**ดูแนวคำตอบ**

1. น้ำหนักรวมประมาณ 1 และ expected return ประมาณ 0.10 ส่วน volatility ประมาณ 0.161328
2. เมื่อเปลี่ยนเป็น \(\gamma=+0.0105013\) คำตอบจะไม่ใช่น้ำหนักชุดที่รายงานและไม่รักษา target/budget ตามระบบสมการเดิม
3. Risky weights รวมประมาณ 1.245108 จึงมี risk-free weight ประมาณ −0.245108 หรือกู้ 24.5108%
4. Relative view เปรียบเทียบขาหนึ่งกับอีกขาโดยไม่มี net exposure ส่วน absolute view เลือกระดับผลตอบแทนของตะกร้าที่มีน้ำหนักรวมหนึ่ง
5. Constraint \(w_1\ge0\) binding ที่ \(w_1=0\) และ multiplier สามารถเป็นบวกได้ ส่วน constraint ที่ยัง slack ต้องมี multiplier ศูนย์
6. \(\mathbf1^\top\Delta\mathbf w=\mathbf1^\top\mathbf w_P-\mathbf1^\top\mathbf w_B=1-1=0\)

## ทำต่อใน Python

[ดาวน์โหลด Notebook ของบท Portfolio Optimization & Black–Litterman](portfolio-optimization.ipynb) เพื่อสร้าง covariance, แก้ target-return portfolio, ตรวจ GMV และ tangency, ทำ GLS whitening, คำนวณ Black–Litterman posterior และเปรียบเทียบ unconstrained กับ long-only solution ตัวอย่างทั้งหมดใช้ Python standard library และรันซ้ำได้

## เอกสารประกอบ

- *Fundamentals of Optimization and Application to Portfolio Selection*, CQF, เอกสาร PDF ที่ผู้ใช้ให้มา, 145 หน้า เนื้อหาหลักของบทนี้เรียบเรียงจากหน้า 4–137 โดยคำนวณสูตรและตัวเลขใหม่ จุดพิมพ์คลาดใน GLS, Lagrange, Black–Litterman และ active weights ได้รับการแก้ก่อนใช้
- Fischer Black and Robert Litterman, “Global Portfolio Optimization,” *Financial Analysts Journal*, 48(5), 28–43, 1992
- Jay Walters, *The Black–Litterman Model in Detail*, working paper, 2011 ใช้เป็นที่มาของตัวอย่าง views ตามรายการอ้างอิงในเอกสารประกอบ

ภาพกราฟและไดอะแกรมในบทสร้างใหม่จากสมการและข้อมูลสมมติด้วย `scripts/make_portfolio_optimization_figures.py` ตาม visual route `no-image-generator` ไม่มีภาพจาก PDF หรือข้อมูลตลาดถูกคัดลอกเข้ามา รายละเอียดที่มา สมมติฐาน และค่าตรวจอยู่ใน [`data/portfolio-optimization-provenance.json`](data/portfolio-optimization-provenance.json)